<a href="https://colab.research.google.com/github/Brandeis-Visual-Analytics/COSI-165b-pas/blob/main/pa2/pa2_training_deeper_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COSI 165b - Deep Learning

Programming Assignment 2

Brandeis University

September 17, 2026

**Check Moodle for Due Date**.  All assignments are due at 11:59 PM on the date listed on Moodle.  Check the syllabus for late submission policy.

**Any AI usage must be reported** at the [unique disclosure form for this PA2](https://genai.cs.brandeis.edu/prompt/bb8face7-c1c7-4af8-af00-c0d51f3bc86f).  AI usage will not affect your grade; we are gathering this information to help us design future instances of the course.

## Watching networks train: loss curves, learning rates, and optimizers

**Name:** _your name_

This is an **individual** assignment. Make sure that you **File -> Save a copy in Drive** first (so your work persists).  Expected behavior for this notebook is that it runs **top to bottom** to produce all answers.  It may be helpful to read along **UDL Chapter 6** as you complete this assignment.

**What you'll do:** train shallow and deep networks on **KMNIST**, learn to read loss curves and choose learning rates and batch sizes, then move to a (subsampled) **CIFAR-10** and compare **optimizers** while watching how the parameters actually change.

**Runtime:** everything is sized for a free Colab instance (about 5 minutes max for any cell). Turn on a GPU: *Runtime -> Change runtime type -> T4 GPU*.

**Submitting:** *File -> Save a copy in Drive* first. Then *Share -> General access: Brandeis University, Viewer -> Copy link*, and paste the link into the PA2 assignment on Moodle. Run all cells so your outputs are visible.

## Part 0 - Setup
Run these cells. They load KMNIST and define a **single, flexible `train()` loop** we reuse for the whole assignment, plus plotting helpers. **Read `train()` closely** - you'll be asked about it later, and the graduate question asks you to modify a copy of it.

In [ ]:
import time, torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X.to(device)).argmax(1) == y.to(device)).float().mean().item()


def train(model, Xtr, ytr, Xval, yval, optimizer, epochs,
          batch_size=None, shuffle=True, track_updates=False):
    """One flexible training loop for the whole assignment.

      batch_size=None -> full-batch gradient descent (one step per epoch)
      optimizer       -> any torch.optim optimizer you pass in
      shuffle         -> reshuffle the example order each epoch
                         (set False to see why we shuffle)
      track_updates   -> also record how much the parameters change each step
    Returns a history dict with per-epoch train/val loss and accuracy.
    Prints one line per epoch so you can see progress while it runs.
    """
    model.to(device)
    Xtr, ytr = Xtr.to(device), ytr.to(device)
    Xval, yval = Xval.to(device), yval.to(device)
    loss_fn = nn.CrossEntropyLoss()
    n = Xtr.shape[0]
    bs = n if batch_size is None else batch_size
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
            "step": [], "upd_p05": [], "upd_p50": [], "upd_p95": []}
    t0 = time.time(); step = 0
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
        for i in range(0, n, bs):
            idx = order[i:i + bs]
            optimizer.zero_grad()
            loss = loss_fn(model(Xtr[idx]), ytr[idx])
            loss.backward()
            if track_updates:
                before = [p.detach().clone() for p in model.parameters()]
            optimizer.step()
            if track_updates:
                deltas = torch.cat([(p.detach() - b).abs().flatten()
                                    for p, b in zip(model.parameters(), before)])
                q = torch.quantile(deltas, torch.tensor([0.05, 0.5, 0.95], device=device))
                hist["step"].append(step)
                hist["upd_p05"].append(q[0].item())
                hist["upd_p50"].append(q[1].item())
                hist["upd_p95"].append(q[2].item())
            step += 1
        model.eval()
        with torch.no_grad():
            hist["train_loss"].append(loss_fn(model(Xtr), ytr).item())
            hist["val_loss"].append(loss_fn(model(Xval), yval).item())
            hist["train_acc"].append((model(Xtr).argmax(1) == ytr).float().mean().item())
            hist["val_acc"].append((model(Xval).argmax(1) == yval).float().mean().item())
        print(f"  epoch {epoch + 1:2d}/{epochs}   train_loss {hist['train_loss'][-1]:.3f}"
              f"   val_loss {hist['val_loss'][-1]:.3f}   val_acc {hist['val_acc'][-1]:.3f}")
    hist["time"] = time.time() - t0
    return hist


def summary(hist, label):
    """One-line end-of-run summary: label (e.g. 'full batch'/'SGD' + hyperparameters), val_acc, time."""
    print(f"[{label}]  ->  val_acc = {hist['val_acc'][-1]:.3f}   time = {hist['time']:.1f}s")


def plot_curves(hist, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
    ax[0].set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss  " + title); ax[0].legend()
    ax[1].plot(hist["train_acc"], label="train"); ax[1].plot(hist["val_acc"], label="val")
    ax[1].set(xlabel="epoch", ylabel="accuracy", title="Accuracy  " + title); ax[1].legend()
    plt.tight_layout(); plt.show()


def plot_param_change(hist, title="parameter change per SGD step"):
    s = hist["step"]
    plt.figure(figsize=(7, 4))
    plt.plot(s, hist["upd_p50"], color="C0", label="median |change|")
    plt.fill_between(s, hist["upd_p05"], hist["upd_p95"], alpha=0.25, color="C0", label="5-95%")
    plt.xlabel("SGD step"); plt.ylabel("|change| per parameter")
    plt.title(title); plt.legend(); plt.show()


def plot_grad_norms(grad_norms, title="per-layer gradient norm"):
    plt.figure(figsize=(8, 4))
    for name, vals in grad_norms.items():
        if name.endswith("weight"):
            plt.plot(vals, label=name)
    plt.xlabel("SGD step"); plt.ylabel("gradient L2 norm")
    plt.title(title); plt.legend(fontsize=8); plt.show()


def count_params(m):
    return sum(p.numel() for p in m.parameters())

In [ ]:
from torchvision import datasets

km = datasets.KMNIST('.', train=True, download=True)
X = (km.data.float() / 255.).reshape(-1, 784)
Y = km.targets
# standardize per pixel (helps make the learning-rate behavior clean)
mu, sd = X.mean(0, keepdim=True), X.std(0, keepdim=True) + 1e-6
X = (X - mu) / sd
# hold out 10k examples as a validation set
Xval_k, yval_k = X[:10000], Y[:10000]
Xtr_k,  ytr_k  = X[10000:], Y[10000:]
print("KMNIST  train:", Xtr_k.shape, " val:", Xval_k.shape)

In [ ]:
# Model factories used throughout Parts 1-3 (KMNIST, 784 inputs, 10 classes).
def make_shallow(width=30):        # 1 hidden layer, K=30
    return nn.Sequential(nn.Linear(784, width), nn.ReLU(),
                         nn.Linear(width, 10))

def make_deep(width=28):           # 3 hidden layers, K=28  -> ~same #params as shallow
    return nn.Sequential(nn.Linear(784, width), nn.ReLU(),
                         nn.Linear(width, width), nn.ReLU(),
                         nn.Linear(width, width), nn.ReLU(),
                         nn.Linear(width, 10))

print("shallow params:", count_params(make_shallow()),
      " | deep params:", count_params(make_deep()))

## Part 1 - A shallow network, full-batch
Start with a **1 hidden layer** network, **K=30** hidden units, trained with **full-batch** gradient descent for **10 epochs**. (With full batch, one epoch = one gradient step, so this is only 10 steps.)

In [ ]:
shallow = make_shallow()
opt = torch.optim.SGD(shallow.parameters(), lr=0.5)
hist = train(shallow, Xtr_k, ytr_k, Xval_k, yval_k, opt, epochs=10, batch_size=None)
plot_curves(hist, "shallow - full batch, 10 epochs")
summary(hist, "full batch, lr=0.5, 10 epochs")

### Q1. Read the curve.
Look at the **training and validation loss together**. Is the model **overfitting**? Why or why not? Do you think it is **done training**? Why or why not? (2-4 sentences.)

> _Your answer here._

## Part 2 - Same network, stochastic gradient descent
Now train the *same* shallow network with **mini-batch SGD**. **You** choose the batch size and learning rate. **Copy the run cell for each (lr, batch_size) you try** and label it, so the notebook documents what you tried. Train as many epochs as you like within reason (keep each run quick).

In [ ]:
# EXAMPLE run - copy this cell for each (lr, batch_size) you try, and label it.  Use the empty cells beneath to try new runs
shallow = make_shallow()
lr, batch_size, epochs = 0.1, 128, 15          # <-- change these and document each run
opt = torch.optim.SGD(shallow.parameters(), lr=lr)
hist = train(shallow, Xtr_k, ytr_k, Xval_k, yval_k, opt,
             epochs=epochs, batch_size=batch_size, shuffle=True)
plot_curves(hist, f"shallow - SGD lr={lr} bs={batch_size}")
summary(hist, f"SGD, lr={lr}, bs={batch_size}, {epochs} epochs")

### Q2A. What did SGD change?
Compared with full-batch in Part 1, what effect did **SGD** have? How did **batch size** and **learning rate** each affect the learning (speed, stability, final accuracy)? Refer to the runs you documented. (3-6 sentences.)

> _Your answer here._

### Q2B. Count the updates.
A parameter **update** is one call to `optimizer.step()`. Roughly **how many updates** happened in your **full-batch** run in Part 1, versus your **SGD** run(s) in Part 2?

*Full batch = one update per epoch. Mini-batch SGD = about `ceil(n_train / batch_size)` updates per epoch, with `n_train = 50000`.*

Could that difference explain **how much the model learned** - and therefore how much **overfitting** was even possible? (2-4 sentences.)

> _Your answer here._

## Part 3 - A deeper network (same parameter budget)
Now use **3 hidden layers**, widened to **K=28** so the deep net has ~the **same number of parameters** as the shallow one (~24k). With the size held fixed, any difference is about **depth** and **how hard the network is to train**, not capacity.

Two things to watch for:

- If the shallow network already does well, adding depth may **not** buy much accuracy - the task may be near a **performance plateau** for this family of models.
- Depth can make optimization **harder**. Watch whether **full-batch** gradient descent struggles to move the deeper network at all (only ~10 updates for many more layers), while **SGD** - many more, noisier updates - trains it to about the **same** accuracy as the shallow net.

Run **full-batch**, then **SGD**, and compare.

In [ ]:
# Build the deep network with make_deep(), then train it FULL-BATCH (batch_size=None)
# for ~10 epochs. Plot the curves and print summary(hist, "deep, full batch, ...").
# (Look back at how Parts 1-2 called train() and summary().)
# Your code here

### Meet the DataLoader

For full-batch training we just hand the whole dataset to `train()` at once. For the **mini-batch** run, this time we're going to use a special PyTorch wrapper object called a **`DataLoader`**.

A `DataLoader` wraps a dataset and hands it back **one mini-batch at a time** as you loop over it (`for xb, yb in loader:`). It's helpful because it does the fiddly bookkeeping for you:

- **Batching** - splits the data into batches of `batch_size` (and gives you a smaller last batch automatically).
- **Shuffling** - reshuffles the example order every epoch when `shuffle=True` (recall from lecture why that matters).
- **Efficiency** - can prepare the next batch in **background worker processes** (`num_workers`) and pin memory for faster GPU transfer, so the GPU isn't left waiting on data.
- **Standard idiom** - it's how essentially every real PyTorch training loop is written, so it's worth getting comfortable with.

*(Our `train()` slices batches by hand so you can see the mechanics; a `DataLoader` is the packaged, production version of the same idea.)*

Docs: [Datasets & DataLoaders (PyTorch tutorial)](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)

In [ ]:
# Now train the deep net with mini-batch SGD using a DataLoader (see the reference above).
# Pieces you'll need:
#   from torch.utils.data import TensorDataset, DataLoader
#   train_loader = DataLoader(TensorDataset(Xtr_k, ytr_k), batch_size=128, shuffle=True)
#   for xb, yb in train_loader:   -> move to device, then zero_grad / forward / loss / backward / step
#   after each epoch, record train & val loss/acc into a hist dict, then plot_curves(hist, ...)
# Your code here

### Q3A. What did depth do?
With the parameter count held roughly fixed, what effect did using a **deeper** network have on **accuracy**? Did **full-batch** and **SGD** behave differently for the deep net - did one barely move while the other trained fine? What does that suggest about depth helping **performance** vs. making **optimization harder**? (3-6 sentences.)

> _Your answer here._

### Q3B. Push it: a small hyperparameter hunt.
Using the playground cells above, experiment with the **width** of the hidden layers, the **batch size**, the **number of epochs**, and the **learning rate**. What is the **best validation accuracy** you can reach? List any other observations about how these knobs affect **training time**, the **shape of the loss curves**, and **overfitting** (the train/val gap). Document the configurations you tried.

> _Your answer here._

In [ ]:
# Q3B playground: experiment here.
# Build a deep MLP (try different hidden-layer WIDTHs) and train it (vary BATCH SIZE,
# EPOCHS, LEARNING RATE) to push val_acc as high as you can. You wrote this training
# code earlier in the assignment - reuse that approach. Copy this cell per configuration.
# Your code here

## Part 4 - A larger, much harder dataset

Now switch to **CIFAR-10**: 32x32 color photos across 10 classes (previewed below), flattened into 3072-vectors for an MLP. This is a genuinely **hard** problem for a fully-connected network - expect accuracy far below what you got on KMNIST. Train it, read the curves, and then we'll ask whether it is still learning.

In [ ]:
# CIFAR-10 via the fast HuggingFace mirror.
# (torchvision downloads CIFAR-10 from cs.toronto.edu, which is painfully slow right
#  now - tens of minutes. The HF mirror serves the same images from a fast CDN. We
#  STREAM it and stop once we have enough per class, so we only pull a small portion.)
!pip install -q datasets
from datasets import load_dataset
import numpy as np

N_TRAIN, N_VAL = 1000, 400          # images per class
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

need = N_TRAIN + N_VAL
buckets = {c: [] for c in range(10)}
for ex in load_dataset("uoft-cs/cifar10", split="train", streaming=True):
    c = ex["label"]
    if len(buckets[c]) < need:
        buckets[c].append(np.asarray(ex["img"], dtype=np.uint8))   # (32, 32, 3) RGB
    if all(len(v) >= need for v in buckets.values()):
        break

# Stack (balanced: N_TRAIN + N_VAL images per class), then split each block into train + val.
imgs   = np.stack([im for c in range(10) for im in buckets[c][:need]])   # (14000, 32, 32, 3)
labels = np.array([c  for c in range(10) for _  in range(need)])
is_train = (np.arange(len(labels)) % need) < N_TRAIN

train_imgs, val_imgs = imgs[is_train], imgs[~is_train]                    # kept for the preview below
ytr_c  = torch.tensor(labels[is_train])
yval_c = torch.tensor(labels[~is_train])

# Flatten to 3072-vectors and standardize with train statistics.
Xtr_c  = torch.tensor(train_imgs, dtype=torch.float32).reshape(-1, 3072) / 255.
Xval_c = torch.tensor(val_imgs,  dtype=torch.float32).reshape(-1, 3072) / 255.
mu_c, sd_c = Xtr_c.mean(0, keepdim=True), Xtr_c.std(0, keepdim=True) + 1e-6
Xtr_c, Xval_c = (Xtr_c - mu_c) / sd_c, (Xval_c - mu_c) / sd_c

def make_cifar_mlp():
    return nn.Sequential(nn.Linear(3072, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(),
                         nn.Linear(128, 10))

print("CIFAR  train:", Xtr_c.shape, " val:", Xval_c.shape)

### Meet CIFAR-10

Unlike KMNIST's 28x28 grayscale characters, CIFAR-10 is **32x32 color photos** across **10 everyday classes** (airplane, automobile, bird, cat, ...). Flattened, each image is a **3072-dimensional** vector - and there's no clean per-pixel template like MNIST had, so a plain MLP will struggle. That's the point: it sets up *why* we'll want convolutional networks later in the course. Here's a sample.

In [ ]:
# A peek at the data: 8 examples per class.
tr_lab = labels[is_train]
fig, axes = plt.subplots(10, 8, figsize=(9, 11))
for c in range(10):
    cls = train_imgs[tr_lab == c]
    for j in range(8):
        ax = axes[c, j]
        ax.imshow(cls[j]); ax.set_xticks([]); ax.set_yticks([])
    axes[c, 0].set_ylabel(CLASS_NAMES[c], rotation=0, ha="right", va="center", fontsize=11)
fig.suptitle("CIFAR-10 - 8 examples per class", y=1.005)
plt.tight_layout(); plt.show()

In [ ]:
# Train the CIFAR MLP on this hard dataset: 30 epochs of mini-batch SGD.
# track_updates=True records how much the parameters move each step (used in Q4b).
net = make_cifar_mlp()
opt = torch.optim.SGD(net.parameters(), lr=0.05)
hist_cifar = train(net, Xtr_c, ytr_c, Xval_c, yval_c, opt,
                   epochs=30, batch_size=128, shuffle=True, track_updates=True)
plot_curves(hist_cifar, "CIFAR MLP - SGD, 30 epochs")
summary(hist_cifar, "CIFAR MLP, SGD, lr=0.05, bs=128, 30 epochs")

### Q4a. Read the curves on a hard problem.
Looking at the training and validation loss/accuracy: is the model **overfitting**? Does it look like it should **train even longer**, or has it leveled off? (2-4 sentences.)

> _Your answer here._

### Q4b. Is it still learning?
We can't look at a million-dimensional loss surface directly, but we can ask a proxy question: **are the parameters still changing?** While the weights keep moving each step, the optimizer is still navigating the loss surface and the model is still learning; once those changes shrink toward zero, training has effectively stalled. The cell below plots how much the parameters change per step. Use it to decide: **does this model need to train more?** (2-4 sentences.)

> _Your answer here._

In [ ]:
# Q4b: how much are the parameters still moving each step?
plot_param_change(hist_cifar)

## Part 5 - Optimizers
Keep the CIFAR MLP fixed and swap the optimizer. Try **SGD + momentum**, **Adam**, and **SGD with weight decay (no momentum)**. For each, one hyperparameter is called out - run it at **2-3 values** and describe the effect.

In [ ]:
# SGD + momentum   (knob to vary: momentum) -- worked example
sgd_mom_net = make_cifar_mlp()
opt = torch.optim.SGD(sgd_mom_net.parameters(), lr=0.05, momentum=0.9)
plot_curves(train(sgd_mom_net, Xtr_c, ytr_c, Xval_c, yval_c, opt, epochs=30, batch_size=128),
            "SGD + momentum=0.9")

# Adam   (knob to vary: lr)
adam_net = make_cifar_mlp()
adam_opt = None      # TODO: make an Adam optimizer -> torch.optim.Adam(adam_net.parameters(), lr=...)  (try lr=1e-3)
plot_curves(train(adam_net, Xtr_c, ytr_c, Xval_c, yval_c, adam_opt, epochs=30, batch_size=128),
            "Adam")

# SGD + weight decay, no momentum   (knob to vary: weight_decay)
weight_decay_net = make_cifar_mlp()
wd_opt = None        # TODO: SGD with weight_decay and no momentum -> torch.optim.SGD(weight_decay_net.parameters(), lr=0.05, weight_decay=...)  (try 1e-3)
plot_curves(train(weight_decay_net, Xtr_c, ytr_c, Xval_c, yval_c, wd_opt, epochs=30, batch_size=128),
            "SGD + weight_decay")

### Q5. Optimizer effects.
For each optimizer, what did you see (convergence speed, stability, final val accuracy)? For the called-out knob - **momentum** (SGD+momentum), **learning rate** (Adam), **weight decay** (SGD) - what effect did changing it have? (4-8 sentences.)

> _Your answer here._

## Part 6 - (Graduate students only) Per-layer gradients
**Copy the `train()` loop** from Part 0 and modify it to record, at each step, the **gradient norm of every layer** into a `grad_norms` dictionary. Then plot them with `plot_grad_norms`. Run it on the **deep** network (Part 3) so there are several layers to compare.

*Hint: the gradients exist only after `loss.backward()` and before the next `zero_grad()`. Iterate `model.named_parameters()` and append `p.grad.norm().item()` for each.*

In [ ]:
deep = make_deep()
grad_norms = {name: [] for name, _ in deep.named_parameters()}

def train_logging_grads(model, Xtr, ytr, optimizer, epochs=3, batch_size=128):
    """A COPY of train(). Add ONE thing where marked: log each layer's grad norm."""
    model.to(device); Xtr, ytr = Xtr.to(device), ytr.to(device)
    loss_fn = nn.CrossEntropyLoss(); n = Xtr.shape[0]
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(Xtr[idx]), ytr[idx])
            loss.backward()
            # TODO: for each named parameter, append its gradient norm to grad_norms[name]
            optimizer.step()

opt = torch.optim.SGD(deep.parameters(), lr=0.1)
train_logging_grads(deep, Xtr_k, ytr_k, opt, epochs=30, batch_size=128)
plot_grad_norms(grad_norms)

### Q6. What do the per-layer gradient norms tell you?
Look at each layer's gradient norm over training. Do the gradients look **healthy** - settling into a stable range rather than shrinking toward zero (**vanishing**) or blowing up (**exploding**)?

Next, let it train for many more epochs.  Do you observe the behavior changing?  Is there anything you can infer about the network and whether there is more for it to learn?

> _Your answer here._